# EV Scheduling

Analysis for the selected simulation run. Each chart is in a separate cell so its inputs, labels, and styling can be adjusted directly.


In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

V_LIMIT = 0.95

NAVY = "#1A204C"
MBLUE = "#085A9B"
GREEN = "#1D772A"
ORANGE = "#FAA449"
RED = "#C00000"
GRAY = "#6B7486"
LIGHT = "#D9DFEA"


In [ ]:
DATA_DIR = r""
RUN_DIR = Path(DATA_DIR) if DATA_DIR else None

if RUN_DIR is None:
    print("Run the simulation and open its Notebook to see the analysis.")


## Data preparation

These helper functions read the selected run and find a matching controlled or uncontrolled companion run when one is available.


In [ ]:
def load_wiring(run_dir: Path) -> dict:
    path = Path(run_dir) / "wiring.json"
    if not path.exists():
        raise FileNotFoundError(f"wiring.json not found in {run_dir}")
    return json.loads(path.read_text(encoding="utf-8"))


def component_types(wiring: dict) -> set[str]:
    return {component.get("type") for component in wiring.get("components", [])}


def component_params(wiring: dict, component_type: str) -> dict:
    for component in wiring.get("components", []):
        if component.get("type") == component_type:
            return component.get("parameters") or {}
    return {}


def _base_bus(column: str) -> str:
    return column.lower().split(".")[0]


def _control_mode(wiring: dict) -> str:
    return str(component_params(wiring, "EVCSComponent").get("control_mode", "unknown"))


def _scenario_sig(wiring: dict) -> str:
    feeder = component_params(wiring, "Feeder")
    ev = dict(component_params(wiring, "EVCSComponent"))
    ev.pop("control_mode", None)
    return json.dumps(
        {"feeder": feeder, "ev": ev},
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def _run_freq_hours(wiring: dict) -> float:
    parameters = component_params(wiring, "Feeder")
    try:
        return float(parameters.get("run_freq_sec", 900.0)) / 3600.0
    except (TypeError, ValueError):
        return 0.25


def _ev_buses(wiring: dict) -> set[str]:
    buses = component_params(wiring, "EVCSComponent").get("evcs_bus") or []
    return {str(bus).lower().split(".")[0] for bus in buses}


def _has_outputs(run_dir: Path) -> bool:
    output_dir = run_dir / "outputs"
    return all(
        (output_dir / filename).exists()
        for filename in ("voltage_real.feather", "voltage_imag.feather", "topology.json")
    )


def _find_companion(run_dir: Path) -> tuple[Path, str] | None:
    try:
        wiring = load_wiring(run_dir)
        current_mode = _control_mode(wiring)
        current_signature = _scenario_sig(wiring)
    except Exception:
        return None

    current_is_uncontrolled = current_mode == "uncontrolled"
    candidates: list[tuple[float, Path, str]] = []
    for other in run_dir.parent.iterdir():
        if not other.is_dir() or other.name == run_dir.name:
            continue
        try:
            other_wiring = load_wiring(other)
            if "EVCSComponent" not in component_types(other_wiring):
                continue
            mode = _control_mode(other_wiring)
            if (mode == "uncontrolled") == current_is_uncontrolled:
                continue
            if _scenario_sig(other_wiring) != current_signature:
                continue
            if not _has_outputs(other):
                continue
            run_record = json.loads((other / "run.json").read_text(encoding="utf-8"))
            if run_record.get("exit_code") != 0:
                continue
            candidates.append((other.stat().st_mtime, other, mode))
        except Exception:
            continue

    if not candidates:
        return None
    candidates.sort(reverse=True)
    _, path, mode = candidates[0]
    return path, mode


In [ ]:
class _RunData:
    def __init__(self, run_dir: Path, label: str):
        self.run_dir = Path(run_dir)
        self.label = label
        output_dir = self.run_dir / "outputs"
        voltage_real = pd.read_feather(output_dir / "voltage_real.feather")
        voltage_imag = pd.read_feather(output_dir / "voltage_imag.feather")
        topology = json.loads((output_dir / "topology.json").read_text(encoding="utf-8"))
        base_voltage = topology["base_voltage_magnitudes"]
        base_by_column = dict(zip(base_voltage["ids"], base_voltage["values"]))

        self.cols = [column for column in voltage_real.columns if column != "time"]
        magnitude = np.sqrt(
            voltage_real[self.cols].to_numpy() ** 2
            + voltage_imag[self.cols].to_numpy() ** 2
        )
        self.pu = magnitude / np.array(
            [base_by_column.get(column, np.nan) for column in self.cols]
        )
        self.finite = np.isfinite(self.pu)
        self.busmin = np.nanmin(np.where(self.finite, self.pu, np.nan), axis=0)

        power_path = output_dir / "powers_real.feather"
        self.demand = None
        self.powers = None
        if power_path.exists():
            power_real = pd.read_feather(power_path)
            power_columns = [column for column in power_real.columns if column != "time"]
            values = power_real[power_columns].to_numpy()
            self.powers = (power_columns, values)
            self.demand = -np.nansum(np.where(values < 0, values, 0), axis=1)

    def ev_voltage(self, ev_buses: set[str]) -> np.ndarray | None:
        indexes = [
            index
            for index, column in enumerate(self.cols)
            if _base_bus(column) in ev_buses
        ]
        if not indexes:
            return None
        return np.nanmin(self.pu[:, indexes], axis=1)

    def ev_load(self, ev_buses: set[str]) -> np.ndarray | None:
        if self.powers is None:
            return None
        power_columns, values = self.powers
        indexes = [
            index
            for index, column in enumerate(power_columns)
            if _base_bus(column) in ev_buses
        ]
        if not indexes:
            return None
        return np.nansum(values[:, indexes], axis=1)

    def peak_demand(self) -> float | None:
        return float(np.nanmax(self.demand)) if self.demand is not None else None


def _feeder_graph(run_dir: Path):
    dss_dir = run_dir / "build" / "feeder" / "opendss"
    coordinates: dict[str, tuple[float, float]] = {}
    bus_coordinates = dss_dir / "Buscoords.dss"
    if not bus_coordinates.exists():
        return None

    for line in bus_coordinates.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) >= 3:
            try:
                coordinates[parts[0].lower()] = (float(parts[1]), float(parts[2]))
            except ValueError:
                pass

    edges: set[tuple[str, str]] = set()
    for filename in ("Lines.dss", "Transformers.dss"):
        path = dss_dir / filename
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
            if filename == "Lines.dss":
                first = re.search(r"bus1=([^\s]+)", line, re.IGNORECASE)
                second = re.search(r"bus2=([^\s]+)", line, re.IGNORECASE)
                pair = [first.group(1), second.group(1)] if first and second else []
            else:
                pair = (
                    re.findall(r"bus=([^\s]+)", line, re.IGNORECASE)
                    if line.lower().startswith("new transformer")
                    else []
                )
            buses = [_base_bus(item) for item in pair]
            for index in range(len(buses) - 1):
                first_bus, second_bus = buses[index], buses[index + 1]
                if (
                    first_bus in coordinates
                    and second_bus in coordinates
                    and first_bus != second_bus
                ):
                    edges.add((first_bus, second_bus))
    return coordinates, edges


def _hours(number_of_steps: int, time_step_hours: float) -> np.ndarray:
    return np.arange(number_of_steps) * time_step_hours


In [ ]:
wiring = None
control_mode = None
ev_buses = set()
time_step_hours = None
this_run = None
companion_run = None
uncontrolled_run = None
controlled_run = None
feeder_graph = None

if RUN_DIR is not None:
    wiring = load_wiring(RUN_DIR)
    control_mode = _control_mode(wiring)
    ev_buses = _ev_buses(wiring)
    time_step_hours = _run_freq_hours(wiring)
    this_run = _RunData(RUN_DIR, "this run")
    feeder_graph = _feeder_graph(RUN_DIR)

    companion = _find_companion(RUN_DIR)
    if companion:
        companion_dir, _companion_mode = companion
        companion_run = _RunData(companion_dir, "companion")

    if companion_run is not None and control_mode == "uncontrolled":
        uncontrolled_run, controlled_run = this_run, companion_run
    elif companion_run is not None:
        uncontrolled_run, controlled_run = companion_run, this_run

    if uncontrolled_run is not None and controlled_run is not None:
        uncontrolled_run.label = "without control"
        controlled_run.label = "with control"




## Feeder voltage map

Bus color shows whether the minimum voltage in this run crosses the voltage limit. Orange rings mark EV buses. This chart is omitted when feeder coordinates are unavailable.


In [ ]:
if RUN_DIR is not None and this_run is not None and feeder_graph is not None:
    coordinates, edges = feeder_graph
    minimum_by_bus: dict[str, float] = {}
    for column, value in zip(this_run.cols, this_run.busmin):
        bus = _base_bus(column)
        minimum_by_bus[bus] = min(minimum_by_bus.get(bus, np.inf), float(value))

    fig, ax = plt.subplots(figsize=(7.4, 5.2))
    for first_bus, second_bus in edges:
        ax.plot(
            [coordinates[first_bus][0], coordinates[second_bus][0]],
            [coordinates[first_bus][1], coordinates[second_bus][1]],
            color=LIGHT,
            linewidth=0.9,
            zorder=1,
        )
    x_coordinates = [coordinates[bus][0] for bus in coordinates]
    y_coordinates = [coordinates[bus][1] for bus in coordinates]
    colors = [
        RED if minimum_by_bus.get(bus, 1.0) < V_LIMIT else GREEN
        for bus in coordinates
    ]
    ax.scatter(
        x_coordinates,
        y_coordinates,
        s=26,
        c=colors,
        edgecolor="white",
        linewidth=0.35,
        zorder=2,
    )
    mapped_ev_buses = [bus for bus in coordinates if bus in ev_buses]
    if mapped_ev_buses:
        ax.scatter(
            [coordinates[bus][0] for bus in mapped_ev_buses],
            [coordinates[bus][1] for bus in mapped_ev_buses],
            s=110,
            facecolor="none",
            edgecolor=ORANGE,
            linewidth=2.0,
            zorder=5,
        )
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(
        "Feeder map — this run's bus minimum voltage (orange ring = EV bus)",
        loc="left",
        fontsize=11,
        color=NAVY,
        fontweight="bold",
    )
    display(fig)
    plt.close(fig)


## Voltage profile

Minimum voltage at each bus-phase measurement point. A matching companion run is included automatically when available.


In [ ]:
if RUN_DIR is not None and this_run is not None:
    if feeder_graph is not None:
        coordinates, edges = feeder_graph
        degree: dict[str, int] = {}
        for first_bus, second_bus in edges:
            degree[first_bus] = degree.get(first_bus, 0) + 1
            degree[second_bus] = degree.get(second_bus, 0) + 1
        hub = max(degree, key=degree.get) if degree else next(iter(coordinates))
        reference_x, reference_y = coordinates[hub]
        distance = np.array(
            [
                math.hypot(
                    coordinates.get(_base_bus(column), (reference_x, reference_y))[0]
                    - reference_x,
                    coordinates.get(_base_bus(column), (reference_x, reference_y))[1]
                    - reference_y,
                )
                for column in this_run.cols
            ]
        )
        order = np.argsort(distance)
    else:
        order = np.argsort(this_run.busmin)[::-1]

    x = np.arange(len(this_run.cols))
    fig, ax = plt.subplots(figsize=(8.6, 4.0))
    ax.axhline(V_LIMIT, color=RED, linestyle="--", linewidth=1.3)
    ax.text(
        len(this_run.cols) * 0.99,
        V_LIMIT + 0.002,
        f"{V_LIMIT} limit",
        ha="right",
        va="bottom",
        fontsize=9,
        color=RED,
        fontweight="bold",
    )
    if (
        companion_run is not None
        and uncontrolled_run is not None
        and controlled_run is not None
        and list(uncontrolled_run.cols) == list(this_run.cols) == list(controlled_run.cols)
    ):
        ax.plot(
            x,
            uncontrolled_run.busmin[order],
            color=GRAY,
            linewidth=1.4,
            label=uncontrolled_run.label,
        )
        ax.plot(
            x,
            controlled_run.busmin[order],
            color=GREEN,
            linewidth=1.8,
            label=controlled_run.label,
        )
    else:
        ax.plot(
            x,
            this_run.busmin[order],
            color=MBLUE,
            linewidth=1.5,
            label=this_run.label,
        )
    ev_indexes = [
        index
        for index, ordered_index in enumerate(order)
        if _base_bus(this_run.cols[ordered_index]) in ev_buses
    ]
    if ev_indexes:
        ax.scatter(
            ev_indexes,
            [this_run.busmin[order][index] for index in ev_indexes],
            s=55,
            color=ORANGE,
            edgecolor=RED,
            linewidth=1.2,
            zorder=5,
            label="EV buses",
        )
    ax.legend(loc="lower left", frameon=False, fontsize=9)
    ax.set_xticks([])
    ax.set_ylabel("min voltage (p.u.)")
    ax.set_xlabel("bus-phase measurement points")
    ax.set_title(
        "Per-bus minimum voltage — with vs without control",
        loc="left",
        fontsize=11,
        color=NAVY,
        fontweight="bold",
    )
    ax.grid(axis="y", color="#EDF0F4", linewidth=0.8)
    display(fig)
    plt.close(fig)


## EV-bus voltage over time

The trace shows the lowest voltage across the configured EV buses at each simulation step.


In [ ]:
if RUN_DIR is not None and this_run is not None:
    current_ev_voltage = this_run.ev_voltage(ev_buses)
    if current_ev_voltage is not None:
        fig, ax = plt.subplots(figsize=(8.6, 3.6))
        if uncontrolled_run is not None and controlled_run is not None:
            uncontrolled_voltage = uncontrolled_run.ev_voltage(ev_buses)
            controlled_voltage = controlled_run.ev_voltage(ev_buses)
            if uncontrolled_voltage is not None:
                ax.plot(
                    _hours(len(uncontrolled_voltage), time_step_hours),
                    uncontrolled_voltage,
                    color=GRAY,
                    linewidth=1.5,
                    label=uncontrolled_run.label,
                )
            if controlled_voltage is not None:
                ax.plot(
                    _hours(len(controlled_voltage), time_step_hours),
                    controlled_voltage,
                    color=GREEN,
                    linewidth=1.8,
                    label=controlled_run.label,
                )
        else:
            ax.plot(
                _hours(len(current_ev_voltage), time_step_hours),
                current_ev_voltage,
                color=MBLUE,
                linewidth=1.6,
                label=this_run.label,
            )
        ax.axhline(V_LIMIT, color=RED, linestyle="--", linewidth=1.3)
        ax.set_xlabel("hours from start")
        ax.set_ylabel("voltage (p.u.)")
        ax.set_title(
            "EV-bus voltage over time (worst EV bus each step)",
            loc="left",
            fontsize=11,
            color=NAVY,
            fontweight="bold",
        )
        ax.legend(loc="lower right", frameon=False, fontsize=9)
        ax.grid(axis="y", color="#EDF0F4", linewidth=0.8)
        display(fig)
        plt.close(fig)


## Feeder demand over time

Total feeder supply is calculated from negative real-power values in the simulation output. This chart is omitted when power output is unavailable.


In [ ]:
if RUN_DIR is not None and this_run is not None and this_run.demand is not None:
    fig, ax = plt.subplots(figsize=(8.6, 3.6))
    if (
        uncontrolled_run is not None
        and controlled_run is not None
        and uncontrolled_run.demand is not None
        and controlled_run.demand is not None
    ):
        ax.plot(
            _hours(len(uncontrolled_run.demand), time_step_hours),
            uncontrolled_run.demand,
            color=GRAY,
            linewidth=1.5,
            label=uncontrolled_run.label,
        )
        ax.plot(
            _hours(len(controlled_run.demand), time_step_hours),
            controlled_run.demand,
            color=GREEN,
            linewidth=1.8,
            label=controlled_run.label,
        )
        uncontrolled_peak = uncontrolled_run.peak_demand()
        controlled_peak = controlled_run.peak_demand()
        ax.set_title(
            f"Feeder demand over time — peak {uncontrolled_peak:.0f} → "
            f"{controlled_peak:.0f} kW "
            f"({100 * (controlled_peak - uncontrolled_peak) / uncontrolled_peak:+.1f} %)",
            loc="left",
            fontsize=11,
            color=NAVY,
            fontweight="bold",
        )
    else:
        ax.plot(
            _hours(len(this_run.demand), time_step_hours),
            this_run.demand,
            color=MBLUE,
            linewidth=1.6,
            label=this_run.label,
        )
        ax.set_title(
            "Feeder demand over time",
            loc="left",
            fontsize=11,
            color=NAVY,
            fontweight="bold",
        )
    ax.set_xlabel("hours from start")
    ax.set_ylabel("feeder supply (kW)")
    ax.legend(loc="upper left", frameon=False, fontsize=9)
    ax.grid(axis="y", color="#EDF0F4", linewidth=0.8)
    display(fig)
    plt.close(fig)


## EV-bus load over time

The comparison highlights charging reduced at the peak and charging shifted to later periods.


In [ ]:
if RUN_DIR is not None and this_run is not None:
    current_ev_load = this_run.ev_load(ev_buses)
    if current_ev_load is not None:
        fig, ax = plt.subplots(figsize=(8.6, 3.6))
        if uncontrolled_run is not None and controlled_run is not None:
            uncontrolled_load = uncontrolled_run.ev_load(ev_buses)
            controlled_load = controlled_run.ev_load(ev_buses)
            if uncontrolled_load is not None:
                ax.plot(
                    _hours(len(uncontrolled_load), time_step_hours),
                    uncontrolled_load,
                    color=GRAY,
                    linewidth=1.5,
                    label=uncontrolled_run.label,
                )
            if controlled_load is not None:
                ax.plot(
                    _hours(len(controlled_load), time_step_hours),
                    controlled_load,
                    color=GREEN,
                    linewidth=1.8,
                    label=controlled_run.label,
                )
            if uncontrolled_load is not None and controlled_load is not None:
                hours = _hours(len(uncontrolled_load), time_step_hours)
                ax.fill_between(
                    hours,
                    controlled_load,
                    uncontrolled_load,
                    where=uncontrolled_load > controlled_load,
                    color=ORANGE,
                    alpha=0.25,
                    linewidth=0,
                    label="charging reduced at peak",
                )
                if bool(np.any(controlled_load > uncontrolled_load + 1e-6)):
                    ax.fill_between(
                        hours,
                        uncontrolled_load,
                        controlled_load,
                        where=controlled_load > uncontrolled_load,
                        color=GREEN,
                        alpha=0.25,
                        linewidth=0,
                        label="charging shifted later",
                    )
        else:
            ax.plot(
                _hours(len(current_ev_load), time_step_hours),
                current_ev_load,
                color=MBLUE,
                linewidth=1.6,
                label=this_run.label,
            )
        ax.set_xlabel("hours from start")
        ax.set_ylabel("EV-bus power (kW)")
        ax.set_title(
            "Load at the EV buses over time (residential + EV charging)",
            loc="left",
            fontsize=11,
            color=NAVY,
            fontweight="bold",
        )
        ax.legend(loc="upper right", frameon=False, fontsize=9)
        ax.grid(axis="y", color="#EDF0F4", linewidth=0.8)
        display(fig)
        plt.close(fig)


## Cumulative EV-bus energy

A late catch-up indicates shifted charging; a persistent gap indicates trimmed charging. This comparison is shown only when a matching companion run is available.


In [ ]:
if (
    RUN_DIR is not None
    and this_run is not None
    and uncontrolled_run is not None
    and controlled_run is not None
):
    current_ev_load = this_run.ev_load(ev_buses)
    uncontrolled_load = uncontrolled_run.ev_load(ev_buses)
    controlled_load = controlled_run.ev_load(ev_buses)
    if (
        current_ev_load is not None
        and uncontrolled_load is not None
        and controlled_load is not None
    ):
        uncontrolled_energy = np.nancumsum(uncontrolled_load) * time_step_hours
        controlled_energy = np.nancumsum(controlled_load) * time_step_hours
        hours = _hours(len(uncontrolled_load), time_step_hours)

        fig, ax = plt.subplots(figsize=(8.6, 3.6))
        ax.plot(
            hours,
            uncontrolled_energy,
            color=GRAY,
            linewidth=1.6,
            label="without control",
        )
        ax.plot(
            hours,
            controlled_energy,
            color=GREEN,
            linewidth=1.9,
            label="with control",
        )
        delivered_percent = (
            100 * controlled_energy[-1] / uncontrolled_energy[-1]
            if uncontrolled_energy[-1]
            else float("nan")
        )
        ax.annotate(
            f"{delivered_percent:.1f} % of uncontrolled energy delivered",
            (hours[-1], controlled_energy[-1]),
            textcoords="offset points",
            xytext=(-8, -14),
            ha="right",
            fontsize=9.5,
            color=GREEN,
            fontweight="bold",
        )
        ax.set_xlabel("hours from start")
        ax.set_ylabel("cumulative EV-bus energy (kWh)")
        ax.set_title(
            "Cumulative EV-bus energy — a late catch-up = shifted charging; "
            "a persistent gap = trimmed charging",
            loc="left",
            fontsize=11,
            color=NAVY,
            fontweight="bold",
        )
        ax.legend(loc="lower right", frameon=False, fontsize=9)
        ax.grid(axis="y", color="#EDF0F4", linewidth=0.8)
        display(fig)
        plt.close(fig)
